# 02 — Transfer Learning (Fine-Tuned Pretrained Model)

**Project:** Bangladeshi Freshwater Fish Image Classification Using CNN and Transfer Learning
**Dataset:** BFF-15 (15 classes)
**Owner:** Mansi Somani

## What this notebook does

1. Loads the frozen train/val/test splits from `data/splits/`.
2. Uses the shared pipeline in `data_utils.py` (no re-splitting, no re-augmentation).
3. Loads a pretrained model (MobileNetV2 by default).
4. Adds a preprocessing layer so the pretrained weights receive the input
   distribution they were trained on.
5. Trains the new head first, then fine-tunes selected base layers at a lower LR.
6. Saves all artifacts into `transfer_learning/`:
   - `transfer_learning/logs/training_log.csv`
   - `transfer_learning/best_model/best.keras`
   - `transfer_learning/plots/`
   - `transfer_learning/metrics/final_metrics.json` and `confusion_matrix.csv`

Notebook 03 will read these to compare against the from-scratch CNN.

## Contract (must not change)

- Class mapping: `CLASS_NAMES` from `data_utils`.
- Splits: loaded from `data_utils.load_splits()`.
- Augmentation: `data_utils.get_train_ds()` only.
- Input normalization: `[0, 1]` from `data_utils`. The pretrained preprocessing
  is applied **inside the model** as a `Lambda` layer.
- Image size: 224 × 224.
- Seed: 42.

In [1]:
import os, sys, json, random
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, optimizers

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from data_utils import (
    get_train_ds, get_val_ds, get_test_ds, get_class_weights,
    CLASS_NAMES, CLASS_TO_IDX, IDX_TO_CLASS,
    IMG_SIZE, BATCH_SIZE, SEED,
)

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)

TensorFlow: 2.21.0


In [2]:
TL_DIR       = PROJECT_ROOT / "transfer_learning"
LOGS_DIR     = TL_DIR / "logs"
PLOTS_DIR    = TL_DIR / "plots"
METRICS_DIR  = TL_DIR / "metrics"
CKPT_DIR     = TL_DIR / "best_model"

for d in [LOGS_DIR, PLOTS_DIR, METRICS_DIR, CKPT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

LOG_CSV      = LOGS_DIR / "training_log.csv"
CKPT_PATH    = CKPT_DIR / "best.keras"
METRICS_JSON = METRICS_DIR / "final_metrics.json"
CM_CSV       = METRICS_DIR / "confusion_matrix.csv"

print("Artifacts folder:", TL_DIR)

Artifacts folder: d:\Dev\College\GenAI\bangladeshi-fish-cnn-classification\transfer_learning


In [3]:
train_ds = get_train_ds()
val_ds   = get_val_ds()
test_ds  = get_test_ds()

print("Train batches:", tf.data.experimental.cardinality(train_ds).numpy())
print("Val   batches:", tf.data.experimental.cardinality(val_ds).numpy())
print("Test  batches:", tf.data.experimental.cardinality(test_ds).numpy())

Train batches: 122
Val   batches: 27
Test  batches: 27


## Model selection

We use **MobileNetV2** as the default pretrained backbone. Alternatives that
work in TensorFlow 2.21 with weights on ImageNet:

- `DenseNet121`
- `EfficientNetB0`
- `ResNet50V2`
- `InceptionV3`

To switch, change `BASE_MODEL` and `PREPROCESS_FN` in the next cell, and use
the matching `preprocess_input` function.

**Important:** `data_utils` gives us `[0, 1]` inputs. Every pretrained model
above expects a different input range. We multiply by 255 and call that
model's `preprocess_input` inside a `Lambda` layer at the top of the network.
This keeps both model notebooks consuming the same data.